🔹 Step 1: Environment Setup

In [94]:
# Install required libraries (run once)
# pip install pandas numpy matplotlib scikit-learn tensorflow

In [95]:
!python --version

Python 3.12.12


🔹 Step 2: Import Libraries

In [96]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Bidirectional
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping

🔹 Step 3: Load & Inspect Dataset

In [97]:
df = pd.read_csv("Job_3_Resource_sentiment.csv")
print(df.columns)

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')


In [98]:
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [99]:
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [100]:
df.rename(columns={'Positive': 'sentiment'}, inplace=True)
df.rename(columns={'im getting on borderlands and i will murder you all ,': 'text'}, inplace=True)

In [101]:
df.columns

Index(['2401', 'Borderlands', 'sentiment', 'text'], dtype='object')

In [102]:
df.sample(5)

,2401,Borderlands,sentiment,text
52318,10587,RedDeadRedemption(RDR),Neutral,Taking a walk in New Austin back then these da...
2423,1624,CallOfDutyBlackopsColdWar,Neutral,These are the results of the Pawn Takes Pawn Z...
72374,11199,TomClancysGhostRecon,Neutral,In this episode It seems with more investigati...
19217,12492,WorldOfCraft,Positive,i'm so bored i might get world of warcraft aga...
31558,7418,LeagueOfLegends,Neutral,Check out the video!


In [103]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   2401         74681 non-null  int64 
 1   Borderlands  74681 non-null  object
 2   sentiment    74681 non-null  object
 3   text         73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [104]:
df = df[['text', 'sentiment']]

In [105]:
df.sample(5)

,text,sentiment
42950,pubg | wtf!!!????,Negative
74587,Heard people are hands with order their Nvidia...,Neutral
40115,A ban for Battlefield 4 player DearonVX has oc...,Irrelevant
19545,2014 Finally got the RhandlerR Horseman's Reig...,Positive
49783,Is 130k bad for my PS4 with FIFA 20?,Negative


In [106]:
df

,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive
...,...,...
74676,Just realized that the Windows partition of my...,Positive
74677,Just realized that my Mac window partition is ...,Positive
74678,Just realized the windows partition of my Mac ...,Positive
74679,Just realized between the windows partition of...,Positive


In [107]:
df.shape

(74681, 2)

In [108]:
df.isnull().sum()

,0
text,686
sentiment,0


In [109]:
print(df['sentiment'].value_counts())

sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [110]:
df.duplicated().sum()

np.int64(4909)

🔹 Step 4: Data Cleaning

In [111]:
df.dropna(inplace=True)

In [112]:
df.isnull().sum()

,0
text,0
sentiment,0


In [113]:
df['text'] = df['text'].astype(str)

In [114]:
df['text']

,text
0,I am coming to the borders and I will kill you...
1,im getting on borderlands and i will kill you ...
2,im coming on borderlands and i will murder you...
3,im getting on borderlands 2 and i will murder ...
4,im getting into borderlands and i can murder y...
...,...
74676,Just realized that the Windows partition of my...
74677,Just realized that my Mac window partition is ...
74678,Just realized the windows partition of my Mac ...
74679,Just realized between the windows partition of...


In [115]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [116]:
df['text']

,text
0,i am coming to the borders and i will kill you...
1,im getting on borderlands and i will kill you all
2,im coming on borderlands and i will murder you...
3,im getting on borderlands and i will murder y...
4,im getting into borderlands and i can murder y...
...,...
74676,just realized that the windows partition of my...
74677,just realized that my mac window partition is ...
74678,just realized the windows partition of my mac ...
74679,just realized between the windows partition of...


🔹 Step 5: Encode Target Labels

In [117]:
encoder = LabelEncoder()
df['sentiment_encoded'] = encoder.fit_transform(df['sentiment'])

print("Label mapping:")
for i, c in enumerate(encoder.classes_):
    print(i, "->", c)

Label mapping:
0 -> Irrelevant
1 -> Negative
2 -> Neutral
3 -> Positive


In [118]:
encoder.classes_

array(['Irrelevant', 'Negative', 'Neutral', 'Positive'], dtype=object)

In [119]:
df.sample(10)

,text,sentiment,sentiment_encoded
35468,so proud you team microsoft taiwan microsoftta...,Positive,3
7592,twitchtvafricanbattler i hit sleep anyways im...,Positive,3
64773,clintoldenburg eamaddennfl i just got a msg po...,Neutral,2
20704,the,Negative,1
22676,for fuck sake this happend to me with a vs on ...,Negative,1
64978,ea eamaddennfl your recent announcement that e...,Negative,1
63115,no way madden eamaddennfl yall see this fix t...,Negative,1
48987,are we more surprised us anymore ea at this po...,Negative,1
7438,i cant wait to play mercy moira morgan and mei...,Positive,3
73246,what,Neutral,2


🔹 Step 6: Text Tokenization & Padding

In [120]:
max_words = 20000
max_length = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text'])

X = pad_sequences(
    tokenizer.texts_to_sequences(df['text']),
    maxlen=max_length,
    padding='post'
)

y = df['sentiment_encoded'].values

In [121]:
tokenizer

In [122]:
df

,text,sentiment,sentiment_encoded
0,i am coming to the borders and i will kill you...,Positive,3
1,im getting on borderlands and i will kill you all,Positive,3
2,im coming on borderlands and i will murder you...,Positive,3
3,im getting on borderlands and i will murder y...,Positive,3
4,im getting into borderlands and i can murder y...,Positive,3
...,...,...,...
74676,just realized that the windows partition of my...,Positive,3
74677,just realized that my mac window partition is ...,Positive,3
74678,just realized the windows partition of my mac ...,Positive,3
74679,just realized between the windows partition of...,Positive,3


In [123]:
X.shape

(73995, 100)

In [124]:
X.dtype

dtype('int32')

In [125]:
y.shape

(73995,)

In [126]:
y.dtype

dtype('int64')

In [127]:
X.ndim

2

In [128]:
y.ndim

1

In [129]:
X.nbytes

29598000

In [130]:
y.nbytes

591960

In [131]:
X

array([[   3,  101,  377, ...,    0,    0,    0],
       [  31,  158,   14, ...,    0,    0,    0],
       [  31,  377,   14, ...,    0,    0,    0],
       ...,
       [  22, 1837,    2, ...,    0,    0,    0],
       [  22, 1837,  693, ...,    0,    0,    0],
       [  22,   32,    2, ...,    0,    0,    0]], dtype=int32)

In [132]:
y

array([3, 3, 3, ..., 3, 3, 3])

🔹 Step 7: Train / Validation / Test Split (MANDATORY)

In [133]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("\nTrain:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)



Train: (51796, 100)
Validation: (11099, 100)
Test: (11100, 100)


Handle Bias (Class Weight)

In [134]:
print(np.bincount(y_train))

[ 9012 15650 12676 14458]


In [135]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print("\nClass Weights:", class_weights_dict)


Class Weights: {0: np.float64(1.4368619618286729), 1: np.float64(0.8274121405750798), 2: np.float64(1.0215367623856106), 3: np.float64(0.8956287176649605)}


In [136]:
print(np.bincount(y_train))

[ 9012 15650 12676 14458]


🔹 Step 8: Build Deep Learning Model (LSTM)

In [137]:
model = Sequential([
    Input(shape=(max_length,)),
    Embedding(max_words, 128),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 100, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,659,332 (10.14 MB)

 Trainable params: 2,659,332 (10.14 MB)

 Non-trainable params: 0 (0.00 B)

🔹 Step 9: Compile Model

In [138]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)